In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 445

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
# Load dataset 
df = pd.read_csv("./datasets/BC-BrainCancer.csv")
df = df.dropna()
print(df.shape)

X = df.drop(columns=["diagnosis"])
y = encoder.fit_transform(df["diagnosis"])

X = pd.get_dummies(X, drop_first=True)
X.reset_index(drop=True, inplace=True)

X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=0.30,shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,test_size=0.50,shuffle=False)

(87, 7)


In [10]:
# Confusion Matrix

def custom_confusion_matrix(y_actual, y_predicted):
    y_true = np.array(y_actual, dtype=int)
    y_pred = np.array(y_predicted, dtype=int)

    num_classes = max(np.max(y_true), np.max(y_pred)) + 1

    confusion_matrix = np.zeros((num_classes, num_classes), dtype=int)

    for actual, predicted in zip(y_true, y_pred):
        confusion_matrix[actual][predicted] += 1

    return confusion_matrix


def custom_accuracy(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)

    correct = np.trace(confusion_matrix)
    total = np.sum(confusion_matrix)

    if total == 0:
        return 0.0
    else:
        return correct / total


def custom_precision(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)

    precision_scores = []

    for i in range(len(confusion_matrix)):
        trueP = confusion_matrix[i][i]
        falseP = np.sum(confusion_matrix[:, i]) - trueP

        if trueP + falseP == 0:
            precision_scores.append(0.0)
        else:
            precision_scores.append(trueP / (trueP + falseP))

    return np.mean(precision_scores)


def custom_recall(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)

    recall_scores = []

    for i in range(len(confusion_matrix)):
        trueP = confusion_matrix[i][i]
        falseN = np.sum(confusion_matrix[i, :]) - trueP

        if trueP + falseN == 0:
            recall_scores.append(0.0)
        else:
            recall_scores.append(trueP / (trueP + falseN))

    return np.mean(recall_scores)


def custom_f1(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)

    f1_scores = []

    for i in range(len(confusion_matrix)):
        trueP = confusion_matrix[i][i]
        falseP = np.sum(confusion_matrix[:, i]) - trueP
        falseN = np.sum(confusion_matrix[i, :]) - trueP

        if trueP + falseP == 0:
            precision = 0.0
        else:
            precision = trueP / (trueP + falseP)

        if trueP + falseN == 0:
            recall = 0.0
        else:
            recall = trueP / (trueP + falseN)

        if precision + recall == 0:
            f1_scores.append(0.0)
        else:
            f1_scores.append((2 * precision * recall) / (precision + recall))

    return np.mean(f1_scores)


def custom_classification_report(y_actual, y_predicted):
    confusion_matrix = custom_confusion_matrix(y_actual, y_predicted)

    return {
        "accuracy": custom_accuracy(y_actual, y_predicted),
        "precision": custom_precision(y_actual, y_predicted),
        "recall": custom_recall(y_actual, y_predicted),
        "f1": custom_f1(y_actual, y_predicted),
        "confusion_matrix": confusion_matrix,
    }

In [9]:
print(df["diagnosis"].unique())
print(df["diagnosis"].value_counts())

<StringArray>
['Meningioma', 'HG glioma', 'LG glioma', 'Other']
Length: 4, dtype: str
diagnosis
Meningioma    42
HG glioma     22
Other         14
LG glioma      9
Name: count, dtype: int64


In [11]:
# Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

DT_Hyperparameters = {
    "max_depth": [3, 5, 7, 10, 12, None],
    "min_samples_split": [2, 5, 10, 20, 30],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "criterion": ["gini", "entropy", "log_loss"],
}

results = []
best = {
    "accuracy": -1,
    "parameters": None,
    "model": None
}

for depth in DT_Hyperparameters["max_depth"]:
    for min_samples_split in DT_Hyperparameters["min_samples_split"]:
        for min_samples_leaf in DT_Hyperparameters["min_samples_leaf"]:
            for criterion in DT_Hyperparameters["criterion"]:
                model = DecisionTreeClassifier(
                    max_depth=depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=SEED
                )
                model.fit(X_train, y_train)
                validation_pred = model.predict(X_val)
                validation_accuracy = custom_accuracy(y_val, validation_pred)
                results.append(
                    (
                        depth,
                        min_samples_split,
                        min_samples_leaf,
                        criterion,
                        validation_accuracy
                    )
                )
                if validation_accuracy > best["accuracy"]:
                    best["accuracy"] = validation_accuracy
                    best["parameters"] = (
                        depth,
                        min_samples_split,
                        min_samples_leaf,
                        criterion
                    )
                    best["model"] = model

results_df = pd.DataFrame(
    results,
    columns=[
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "criterion",
        "val_accuracy"
    ]
).sort_values(
    by="val_accuracy",
    ascending=False
).reset_index(drop=True)

test_pred = best["model"].predict(X_test)
test_report = custom_classification_report(y_test, test_pred)

print(best)
print(results_df)
print(test_report)

{'accuracy': np.float64(0.6923076923076923), 'parameters': (5, 2, 5, 'entropy'), 'model': DecisionTreeClassifier(criterion='entropy', max_depth=5, min_samples_leaf=5,
                       random_state=445)}
     max_depth  min_samples_split  min_samples_leaf criterion  val_accuracy
0          NaN                 10                 5  log_loss      0.692308
1          NaN                 10                 5   entropy      0.692308
2          NaN                  5                 5  log_loss      0.692308
3          NaN                  5                 5   entropy      0.692308
4          NaN                  2                 5  log_loss      0.692308
..         ...                ...               ...       ...           ...
445        5.0                 10                10      gini      0.384615
446        3.0                 20                10      gini      0.384615
447        5.0                  2                10      gini      0.384615
448        3.0                 

In [ ]:
#Using XGBoost Classifier
XGB_Hyperparameters = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}
random_number_generator = np.random.RandomState(SEED)
keys = list(XGB_Hyperparameters.keys())

all_combinations = []
results = []
best = {
    "accuracy": -1,
    "parameters": None,
    "model": None
}

for i in range(180):
    combination = {
        key: XGB_Hyperparameters[key][random_number_generator.randint(len(XGB_Hyperparameters[key]))]
        for key in keys
    }
    all_combinations.append(tuple(sorted(combination.items())))
unique_combinations = list(dict.fromkeys(all_combinations))[:60]

for combination in unique_combinations:
    params = dict(combination)
    model = XGBClassifier(
        objective="binary:logistic",
        random_state=SEED,
        n_jobs=4,
        verbosity=0,
        eval_metric="logloss",
        **params
    )

    model.fit(X_train, y_train)
    validation_pred = model.predict(X_val)
    validation_accuracy = custom_accuracy(y_val, validation_pred)

    results.append(
        (
            params["max_depth"],
            params["learning_rate"],
            params["n_estimators"],
            params["subsample"],
            params["colsample_bytree"],
            params["min_child_weight"],
            params["gamma"],
            validation_accuracy
        )
    )

    if validation_accuracy > best["accuracy"]:
        best["accuracy"] = validation_accuracy
        best["parameters"] = params
        best["model"] = model

results_df = pd.DataFrame(
    results,
    columns=[
        "max_depth",
        "learning_rate",
        "n_estimators",
        "subsample",
        "colsample_bytree",
        "min_child_weight",
        "gamma",
        "val_accuracy"
    ]
).sort_values(
    by="val_accuracy",
    ascending=False
).reset_index(drop=True)

test_pred = best["model"].predict(X_test)
test_report = custom_classification_report(y_test, test_pred)

print(best)
print(results_df)
print(test_report)